# 12 — Construct: Ottimizzazione iperparametri Random Forest

**Fase PACE**: Construct  
**Obiettivo**: ottimizzare gli iperparametri del modello Random Forest 
costruito nella fase Analyze (Q7.2) attraverso due approcci a confronto:
- **Approccio A**: GridSearchCV su dataset completo (~2 ore)
- **Approccio B**: RandomizedSearchCV su campione 20% del dataset (~20 min)


**Baseline** (dal Blocco 7):
- Parametri: `n_estimators=100`, `max_depth=10`, `class_weight='balanced'`
- Accuracy: 71%, Recall Cleared: 82%, Precision Cleared: 45%

**Input**: `data/processed/crimes_features.parquet`

In [7]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.utils import resample
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_parquet('../../data/processed/crimes_features.parquet')
df.head()

,DR_NO,Date Rptd,DATE OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,...,LON,hour_occ,year,month,day_of_week,hour_bins,age_group,crime_category,report_delay,is_domestic
0,1307355,2010-02-20,2010-02-20,13,Newton,1385,2,900,VIOLATION OF COURT ORDER,0913 1814 2000,...,-118.2695,13,2010,2,Saturday,Afternoon,Adult,person,0,False
1,11401303,2010-09-13,2010-09-12,14,Pacific,1485,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",0329,...,-118.3962,0,2010,9,Sunday,Night,NaN,property,1,False
2,70309629,2010-08-09,2010-08-09,13,Newton,1324,2,946,OTHER MISCELLANEOUS CRIME,0344,...,-118.2524,15,2010,8,Monday,Afternoon,NaN,other,0,False
3,90631215,2010-01-05,2010-01-05,6,Hollywood,646,2,900,VIOLATION OF COURT ORDER,1100 0400 1402,...,-118.3295,1,2010,1,Tuesday,Night,Adult,person,0,False
4,100100501,2010-01-03,2010-01-02,1,Central,176,1,122,"RAPE, ATTEMPTED",0400,...,-118.2488,21,2010,1,Saturday,Late Night,Adult,person,1,False


In [4]:
# Selezione colonne
q72_df = df[['AREA NAME', 'crime_category', 'Weapon Desc', 'Vict Sex', 
             'age_group', 'Vict Descent', 'report_delay', 
             'hour_bins', 'day_of_week', 'Status Desc']].copy()

# Creazione target
cleared_statuses = ['Adult Arrest', 'Juv Arrest', 'Adult Other', 'Juv Other']
q72_df['is_cleared'] = q72_df['Status Desc'].isin(cleared_statuses).astype(int)
q72_df = q72_df.drop(columns=['Status Desc'])

# Rimozione NaN
q72_df = q72_df.dropna()

feature_cols = ['AREA NAME', 'crime_category', 'Weapon Desc', 'Vict Sex',
                'age_group', 'Vict Descent', 'report_delay', 
                'hour_bins', 'day_of_week']

X = pd.get_dummies(q72_df[feature_cols], drop_first=True)
y = q72_df['is_cleared']

print(f'Features: {X.shape[1]}')
print(f'Campioni: {X.shape[0]}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} campioni')
print(f'Test: {X_test.shape[0]:,} campioni')

Features: 138
Campioni: 2447032
Train: 1,957,625 campioni
Test: 489,407 campioni


In [6]:
X_train_sample, y_train_sample = resample(
    X_train, y_train, 
    n_samples=int(len(X_train) * 0.2),
    random_state=42,
    stratify=y_train
)

print(f'Campione train: {X_train_sample.shape[0]:,}')
print(f'Distribuzione: {y_train_sample.value_counts().to_dict()}')

Campione train: 391,525
Distribuzione: {0: 295710, 1: 95815}


In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='recall',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_random.fit(X_train_sample, y_train_sample)
print(f'Parametri ottimali: {rf_random.best_params_}')
print(f'Miglior recall (CV): {rf_random.best_score_:.4f}')

Fitting 5 folds for each of 20 candidates, totalling 100 fits
